# 0.9 Provincial Network Builder

This notebook demonstrates the use of **Module 12: ProvincialNetworkBuilder** to create road network graphs and distance matrices for a single province.

**Key Features:**
- ARM-Compatible (NetworkX-only, no igraph)
- Builds distance graph (road networks) + beneficiary graph (student flows)
- Designed for regional merging (consistent coordinate precision, boundary nodes)

**Example Province:** Bulacan (PH03014)

**Outputs:**
1. Distance matrix (CSV)
2. Distance graph (GraphML)
3. Beneficiary graph (GraphML)
4. Summary statistics (JSON)
5. Visualizations

## 0. Setup

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from importlib import reload
import matplotlib.pyplot as plt
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import config
from config import setup_notebook, get_path
setup_notebook()

✓ Project root: /workspace/project_paaral
✓ Working directory: /workspace/project_paaral
✓ Python path updated


{'project_root': PosixPath('/workspace/project_paaral'),
 'data': PosixPath('/workspace/project_paaral/data'),
 'modules': PosixPath('/workspace/project_paaral/modules'),
 'notebooks': PosixPath('/workspace/project_paaral/notebooks'),
 'output': PosixPath('/workspace/project_paaral/output'),
 'psgc_shapefiles': PosixPath('/workspace/project_paaral/data/philippines-psgc-shapefiles/dist')}

In [2]:
# Import NodeTableBuilder
from modules import provincial_network_builder
reload(provincial_network_builder)

# Import the module
from modules.provincial_network_builder import ProvincialNetworkBuilder

## 1. Load Data

Load the validated node tables and beneficiary edges from previous modules.

**Note:** Node tables from Module 11 (NodeTableBuilder) should already include the `adm2_pcode` column via spatial join with PSGC geodata. This column is essential for filtering schools to provinces.

In [3]:
# Load public and private node tables (from Module 11)
public_nodes = gpd.read_file('output/public_nodes_valid.gpkg')
private_nodes = gpd.read_file('output/private_nodes_valid.gpkg')

print(f"Public schools loaded: {len(public_nodes):,}")
print(f"Private schools loaded: {len(private_nodes):,}")

Public schools loaded: 44,899
Private schools loaded: 9,305


In [4]:
# Load beneficiary edges (from Module 11)
beneficiary_edges = pd.read_csv('output/beneficiary_edges_valid.csv')

# Ensure string IDs
beneficiary_edges['school_id_origin'] = beneficiary_edges['school_id_origin'].astype(str)
beneficiary_edges['school_id_destination'] = beneficiary_edges['school_id_destination'].astype(str)

print(f"Beneficiary edges loaded: {len(beneficiary_edges):,}")
print(f"Unique origins: {beneficiary_edges['school_id_origin'].nunique():,}")
print(f"Unique destinations: {beneficiary_edges['school_id_destination'].nunique():,}")

Beneficiary edges loaded: 220,777
Unique origins: 34,411
Unique destinations: 2,955


In [5]:
# Verify adm2_pcode column exists
if 'adm2_pcode' in public_nodes.columns:
    print("✓ adm2_pcode column found in node tables")
    print(f"  Public nodes with adm2_pcode: {public_nodes['adm2_pcode'].notna().sum():,} / {len(public_nodes):,}")
    print(f"  Private nodes with adm2_pcode: {private_nodes['adm2_pcode'].notna().sum():,} / {len(private_nodes):,}")
else:
    print("❌ ERROR: adm2_pcode column missing from node tables!")
    print("   Please regenerate node tables using notebook 0.7 with Module 11.")
    print("   Module 11 automatically adds adm2_pcode via spatial join.")

✓ adm2_pcode column found in node tables
  Public nodes with adm2_pcode: 44,899 / 44,899
  Private nodes with adm2_pcode: 9,305 / 9,305


## 2. Select Province

Filter data to a single province. We'll use **Bulacan (PH03014)** as an example.

In [6]:
# Province selection
PROVINCE_CODE = 'PH03014'
PROVINCE_NAME = 'bulacan'

# Road network path
ROAD_NETWORK_PATH = f'output/province_road_networks/{PROVINCE_CODE}_{PROVINCE_NAME}.geojsonl'

# Consolidated geodata path (for boundary identification)
GEODATA_PATH = 'output/consolidated_geodata_matched.gpkg'

In [7]:
# Filter schools to province
public_bulacan = public_nodes[public_nodes['adm2_pcode'] == PROVINCE_CODE].copy()
private_bulacan = private_nodes[private_nodes['adm2_pcode'] == PROVINCE_CODE].copy()

print(f"Bulacan schools:")
print(f"  Public: {len(public_bulacan):,}")
print(f"  Private: {len(private_bulacan):,}")
print(f"  Total: {len(public_bulacan) + len(private_bulacan):,}")

Bulacan schools:
  Public: 681
  Private: 204
  Total: 885


In [8]:
# Filter beneficiary edges to province
# Include edges where origin OR destination is in Bulacan
bulacan_school_ids = set(public_bulacan['school_id']) | set(private_bulacan['school_id'])

bulacan_edges = beneficiary_edges[
    beneficiary_edges['school_id_origin'].isin(bulacan_school_ids) |
    beneficiary_edges['school_id_destination'].isin(bulacan_school_ids)
].copy()

print(f"\nBulacan beneficiary edges: {len(bulacan_edges):,}")
print(f"  In-province flows: {bulacan_edges[(bulacan_edges['school_id_origin'].isin(bulacan_school_ids)) & (bulacan_edges['school_id_destination'].isin(bulacan_school_ids))].shape[0]:,}")
print(f"  Cross-province flows: {bulacan_edges[~((bulacan_edges['school_id_origin'].isin(bulacan_school_ids)) & (bulacan_edges['school_id_destination'].isin(bulacan_school_ids)))].shape[0]:,}")


Bulacan beneficiary edges: 8,568
  In-province flows: 2,401
  Cross-province flows: 6,167


In [9]:
# Check if road network file exists
if not Path(ROAD_NETWORK_PATH).exists():
    print(f"⚠️  Road network file not found: {ROAD_NETWORK_PATH}")
    print(f"Please ensure Module 9 has been run to extract provincial road networks.")
else:
    print(f"✓ Road network file found: {ROAD_NETWORK_PATH}")

✓ Road network file found: output/province_road_networks/PH03014_bulacan.geojsonl


## 3. Initialize Network Builder

Create the ProvincialNetworkBuilder instance with all required data.

In [10]:
# Initialize builder
builder = ProvincialNetworkBuilder(
    province_code=PROVINCE_CODE,
    province_name=PROVINCE_NAME,
    public_nodes_gdf=public_bulacan,
    private_nodes_gdf=private_bulacan,
    beneficiary_edges_df=bulacan_edges,
    road_network_path=ROAD_NETWORK_PATH,
    consolidated_geodata_path=GEODATA_PATH,
    verbose=True
)

INFO:modules.provincial_network_builder:ProvincialNetworkBuilder initialized for bulacan (PH03014)
INFO:modules.provincial_network_builder:  Public schools: 681
INFO:modules.provincial_network_builder:  Private schools: 204
INFO:modules.provincial_network_builder:  Total schools: 885
INFO:modules.provincial_network_builder:  Beneficiary edges: 8,568


## 4. Build Complete Network

Execute the complete network building workflow:
1. Load road network
2. Snap schools to network
3. Build spatial index
4. Compute distance matrix (parallel)
5. Build distance and beneficiary graphs
6. Identify boundary nodes

In [10]:
# Build network
# Parameters:
#   buffer_distance_m: Search radius around each school (5km)
#   max_distance_km: Maximum road distance to consider (15km)
#   n_processes: Number of parallel processes (None = use all CPUs)

results = builder.build_complete_network(
    buffer_distance_m=5000,
    max_distance_km=15,
    n_processes=10  # Adjust based on your CPU
)

INFO:modules.provincial_network_builder:======================================================================
INFO:modules.provincial_network_builder:Building complete network for bulacan
INFO:modules.provincial_network_builder:======================================================================
INFO:modules.provincial_network_builder:
[1/6] Loading road network...
INFO:modules.provincial_network_builder:  Reading GeoJSONL: output/province_road_networks/PH03014_bulacan.geojsonl
INFO:modules.provincial_network_builder:  Loaded 51,760 road segments
INFO:modules.provincial_network_builder:  Converting to NetworkX graph (EPSG:4326)...
INFO:modules.provincial_network_builder:  Projecting to EPSG:3123 for distance calculations...
INFO:modules.provincial_network_builder:  ✓ NetworkX graph: 283,684 nodes, 301,705 edges
INFO:modules.provincial_network_builder:
[2/6] Snapping schools to road network...
INFO:modules.provincial_network_builder:  Snapping 885 schools to network...
INFO:modules.p

KeyboardInterrupt: 

Process ForkPoolWorker-22:
Process ForkPoolWorker-25:
active_threads = {thread.ident for thread in threading.enumerate()}
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 812, in <setcomp>
    active_threads = {thread.ident for thread in threading.enumerate()}
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 
Process ForkPoolWorker-23:
Process ForkPoolWorker-28:
Process ForkPoolWorker-26:
Process ForkPoolWorker-27:
Process ForkPoolWorker-24:
Process ForkPoolWorker-29:
Process ForkPoolWorker-30:
Process ForkPoolWorker-31:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/loc

## 5. Analyze Results

Examine the generated distance matrix and graphs.

In [ ]:
# Access results
distance_matrix = results['distance_matrix']
distance_graph = results['distance_graph']
beneficiary_graph = results['beneficiary_graph']
road_network = results['road_network']
school_mappings = results['school_mappings']
boundary_nodes = results['boundary_nodes']
statistics = results['statistics']

In [ ]:
# Display summary statistics
print("="*70)
print("NETWORK SUMMARY")
print("="*70)
print(json.dumps(statistics, indent=2))

In [ ]:
# Distance matrix statistics
print("\n" + "="*70)
print("DISTANCE MATRIX STATISTICS")
print("="*70)
print(f"Shape: {distance_matrix.shape}")
print(f"\nDistance statistics (meters):")
print(f"  Mean: {distance_matrix.mean().mean():,.0f}m ({distance_matrix.mean().mean()/1000:.1f}km)")
print(f"  Median: {distance_matrix.median().median():,.0f}m ({distance_matrix.median().median()/1000:.1f}km)")
print(f"  Min: {distance_matrix.min().min():,.0f}m ({distance_matrix.min().min()/1000:.1f}km)")
print(f"  Max: {distance_matrix.max().max():,.0f}m ({distance_matrix.max().max()/1000:.1f}km)")
print(f"\nSparsity:")
print(f"  Valid pairs: {distance_matrix.notna().sum().sum():,}")
print(f"  Total pairs: {distance_matrix.size:,}")
print(f"  Density: {distance_matrix.notna().sum().sum() / distance_matrix.size * 100:.2f}%")

In [ ]:
# Sample of distance matrix
print("\nSample of distance matrix (first 5x5):")
display(distance_matrix.iloc[:5, :5])

In [ ]:
# Graph statistics
print("\n" + "="*70)
print("GRAPH STATISTICS")
print("="*70)

print("\nDistance Graph:")
print(f"  Nodes: {distance_graph.number_of_nodes():,}")
print(f"  Edges: {distance_graph.number_of_edges():,}")
print(f"  Density: {nx.density(distance_graph):.6f}")
print(f"  Is connected: {nx.is_weakly_connected(distance_graph)}")

print("\nBeneficiary Graph:")
print(f"  Nodes: {beneficiary_graph.number_of_nodes():,}")
print(f"  Edges: {beneficiary_graph.number_of_edges():,}")
print(f"  Density: {nx.density(beneficiary_graph):.6f}")
print(f"  Total beneficiaries: {sum(nx.get_edge_attributes(beneficiary_graph, 'beneficiary_count').values()):,}")

In [ ]:
# Check boundary nodes
print("\n" + "="*70)
print("BOUNDARY NODES (for regional merging)")
print("="*70)
print(f"Boundary nodes identified: {len(boundary_nodes):,}")
print(f"  ({len(boundary_nodes) / road_network.number_of_nodes() * 100:.1f}% of road network nodes)")

## 6. Visualizations

In [ ]:
# Distance distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
distances_flat = distance_matrix.values.flatten()
distances_flat = distances_flat[~np.isnan(distances_flat)]

axes[0].hist(distances_flat / 1000, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Distance (km)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Road Distances')
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(distances_flat / 1000, vert=True)
axes[1].set_ylabel('Distance (km)')
axes[1].set_title('Road Distance Box Plot')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# School locations
fig, ax = plt.subplots(figsize=(12, 10))

# Plot schools
public_bulacan.plot(ax=ax, color='blue', markersize=10, alpha=0.6, label='Public Schools')
private_bulacan.plot(ax=ax, color='red', markersize=10, alpha=0.6, label='Private Schools')

ax.set_title(f'Schools in {PROVINCE_NAME.title()} Province', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Beneficiary flow statistics
beneficiary_counts = nx.get_edge_attributes(beneficiary_graph, 'beneficiary_count')
beneficiary_values = list(beneficiary_counts.values())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(beneficiary_values, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0].set_xlabel('Beneficiary Count per Edge')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Beneficiary Flows')
axes[0].grid(alpha=0.3)

# Log scale
axes[1].hist(beneficiary_values, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Beneficiary Count per Edge')
axes[1].set_ylabel('Frequency (log scale)')
axes[1].set_title('Distribution of Beneficiary Flows (Log Scale)')
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Export Results

Export all results to the output directory for use in regional merging or further analysis.

In [ ]:
# Export all results
output_dir = Path('output/provincial_networks')
output_dir.mkdir(parents=True, exist_ok=True)

builder.export_all(str(output_dir))

print("\n" + "="*70)
print("EXPORTED FILES")
print("="*70)
for file in sorted(output_dir.glob(f"{PROVINCE_CODE}_{PROVINCE_NAME}*")):
    print(f"  {file.name}")

## 8. Graph Analysis Examples

Demonstrate basic NetworkX graph analysis operations.

In [ ]:
# Example 1: Find shortest path between two schools
# Pick two random schools
sample_schools = distance_matrix.index[:2].tolist()
if len(sample_schools) >= 2:
    origin = sample_schools[0]
    destination = sample_schools[1]
    
    if nx.has_path(distance_graph, origin, destination):
        path = nx.shortest_path(distance_graph, origin, destination, weight='distance_m')
        path_length = nx.shortest_path_length(distance_graph, origin, destination, weight='distance_m')
        
        print("Example Shortest Path:")
        print(f"  From: {origin}")
        print(f"  To: {destination}")
        print(f"  Path length: {path_length:,.0f}m ({path_length/1000:.1f}km)")
        print(f"  Path: {' → '.join(path[:5])}..." if len(path) > 5 else f"  Path: {' → '.join(path)}")

In [ ]:
# Example 2: Degree centrality in beneficiary graph
in_degree = dict(beneficiary_graph.in_degree())
out_degree = dict(beneficiary_graph.out_degree())

# Top 5 schools by incoming beneficiary flows (destinations)
top_destinations = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 Destination Schools (by incoming edge count):")
for school_id, degree in top_destinations:
    print(f"  {school_id}: {degree} incoming edges")

# Top 5 schools by outgoing beneficiary flows (origins)
top_origins = sorted(out_degree.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 Origin Schools (by outgoing edge count):")
for school_id, degree in top_origins:
    print(f"  {school_id}: {degree} outgoing edges")

In [ ]:
# Example 3: Total beneficiaries by school
# Sum beneficiary counts for each school
beneficiary_in = {}
beneficiary_out = {}

for origin, dest, data in beneficiary_graph.edges(data=True):
    count = data.get('beneficiary_count', 0)
    beneficiary_out[origin] = beneficiary_out.get(origin, 0) + count
    beneficiary_in[dest] = beneficiary_in.get(dest, 0) + count

# Top 5 by total beneficiaries received
top_receivers = sorted(beneficiary_in.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 Schools by Total Beneficiaries Received:")
for school_id, count in top_receivers:
    print(f"  {school_id}: {count:,} beneficiaries")

# Top 5 by total beneficiaries sent
top_senders = sorted(beneficiary_out.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 Schools by Total Beneficiaries Sent:")
for school_id, count in top_senders:
    print(f"  {school_id}: {count:,} beneficiaries")

## Summary

This notebook demonstrated:
1. ✓ Loading validated node tables and beneficiary edges
2. ✓ Filtering data to a single province (Bulacan)
3. ✓ Building complete provincial network (distance + beneficiary graphs)
4. ✓ Analyzing network statistics
5. ✓ Visualizing results
6. ✓ Exporting merge-friendly outputs
7. ✓ Basic NetworkX graph analysis

**Next Steps:**
- Repeat for other provinces
- Module 13: Regional network builder (merge provincial networks)
- Discrete choice modeling using distance and beneficiary graphs